## Anti Aliasing - Texture Mapping

#### Objective: 

- $\text{Optimise the calculation of averaging for the Minification problem}$


### Apply Anti-Aliasing in Texture Mapping

### In Regular Averaging we:

- Look at the neighborhood around texel in (u,v) space, and average the colors of all neighbors to ge the textured mapped pixel color.
- To simplify this calculation we usually approximate the neighborhood to a rectangle boundary (shape).

<p align="center">
<img src="image_U12/Screenshot 2025-06-30 at 9.33.56.png" height="300" width="400"/>
</p>

### The problem we're trying to solve
We have a texture image I, and we render a scene where this texture is mapped to some surface when we render the scene a pixel in the rendered image can cover some region on the texture. 
We approximate this region using a rectangle R on the texture I and we need to calculate the average color of all the texels in the rectangle R. 


##### Summed Area Table

$\text{Summed Area table := Allows to calculate the average of any sub -rectangle of the texture in O(1)}$

#### Finding the average color (R, G, B) in Rectangle R (m' by n')

$$ \color{red}r = \frac{1}{m' \cdot n'}\sum_{(i,j)\in R}I_{red}(i,j) $$
$$ \color{green}g = \frac{1}{m' \cdot n'}\sum_{(i,j)\in R}I_{green}(i,j) $$
$$ \color{blue}b = \frac{1}{m' \cdot n'}\sum_{(i,j)\in R}I_{blue}(i,j) $$

<p align="center">
<img src="image_U12/Screenshot 2025-06-30 at 10.05.57.png" height="400" width="400"/>
</p>


- $\text{Time Complexity: O(nm)}$
- $\text{This is out bottleneck}$
- $\text{Given a 2D table I of MxN dimensions, that holds some scalar values I(i,j),} \\ \text{the summed area table T of I is a 2D table of the same dimension MxN, that can support the calculation of the sum of any sub-rectangle in I in constant time.}$
$\text{This means we create a data structure 2D array T, which is derived from an original 2D array I (of MxN), where each element T(i,j) stores the }$ **sum of all values int he rectangle from top-left (0,0) to (i,j)** $\text{in the original table}$

$$T(i,j) = \sum_{x=0}^{i} \sum_{y=0}^{j} I(x, y)$$


<p align="center">
<img src="image_U12/Screenshot 2025-06-30 at 10.05.10.png" height="200" width="600"/>
</p>

- $\text{We can build this table in one passing (i.e. in linear time  size the table itself)}$

$$ T(i,j) = I(i,j) + T(i-1,j) + T(i, j-1) - T(i-1, j-1)$$

- $\text{Two things to note in this equation:}$
1. $\text{For any value T(i,j), which falls outside of the table the default is 0}$
2. $\text{The equation uses the principle of inclusion exclusion to calculate the area}$

#### Current Conclusion

- $\text{By the definition of the table T, we can explicitely find the area between T(0,0) to T(i,j) for any i and j}$
- $\text{To find the sum average area in constant of any areas:}$
$$ Let: \\
	(x_1, y_1) = \text{top-left corner of your subrectangle} \\ 
	(x_2, y_2) = \text{bottom-right corner} \\
\text{(where } x_2 \ge x_1 \text{ and } y_2 \ge y_1 ) \\

\text{Then the sum of values in that rectangle is: } \\ 

\text{sum} = T(x_2, y_2) - T(x_1-1, y_2) - T(x_2, y_1-1) + T(x_1-1, y_1-1)
$$
<p align="center">
<img src="image_U12/Screenshot 2025-06-30 at 9.53.51.png" height="300" width="600"/>
</p>

### Combining with texture Mapping

- $\text{The storage space need to store the summed-area table is more than double the size of the original texture, since this is a summation table the values have a much larger range than the pixel color values, hence it's more than double}$
- $\text{In addition, this method still needs some calcualtions during the rendering time:}$
    - $\text{For every pixel drawn instead of using the lookup table, we must first caluclate the average color values of its neigborhood, based on the summed area and then map}$

- $\text{We want to conserve the original proceedure of the texture mapping and simply use a lookup table for the color values of the pixels, without calculations, even with the minification problem}$
- $\text{This leads us to the following method.}$


#####  MIP-MAP

$\text{MIP-MAP := precomputes several texture for even faster rendering, by allowing the rendering pipeline to continue using the textures as a simple lookup tables, whilst still dealing with aliasin artifacts}$

- 4$\text{We use a sequence of pre-calculated textures, each one is smaller than the previous one (i.e. obtain different sizes of the same type of texture image), usually half the size intervals}$
- $\text{We do this by smoothing and averaging the larger textures, so that they don't contain artifacts themselves}$
- $\text{Depending ont he size of the region that texture mapping is mapped to the final image, we'll use the best size texture to reduce the minification problem the mapping.}

$\text{Space Complexity := Instead of single texture image, we have many images to choose from, and since the size difference is of increments of a } \frac{1}{4} \text{ It won't take more than a third of the original text (in TOTAL)}$

<p align="center">
<img src="image_U12/Screenshot 2025-06-30 at 10.17.02.png" height="400" width="400"/>
<img src="image_U12/Screenshot 2025-06-30 at 10.17.55.png" height="400" width="400"/>
</p>


## 📊 Comparison: Summed Area Table vs MIP-MAP

| **Aspect**              | **Summed Area Table (SAT)**                                                                                       | **MIP-MAP**                                                                                   |
|-------------------------|--------------------------------------------------------------------------------------------------------------------|-----------------------------------------------------------------------------------------------|
| **Definition**          | Precomputes cumulative sums over a 2D image.                                                                      | Creates a pyramid of smaller versions of the texture.                                         |
| **Purpose**             | Enables constant-time averaging of rectangular regions.                                                           | Enables fast minification and reduces aliasing.                                               |
| **Precomputation**      | Builds a 2D table T from original image I:                                                                        | Each level is half the size of the previous (e.g., 512×512 → 256×256).                        |
|                         | $T(i,j) = \sum_{x=0}^{i} \sum_{y=0}^{j} I(x, y)$                                                                   | Each pixel in lower level is average of 2×2 block in higher level.                            |
| **Query Cost**          | $O(1)$ per rectangle:                                                                                              | $O(1)$ lookup from appropriate mip level (or interpolated).                                   |
|                         | $sum = T(x_2,y_2) - T(x_1-1,y_2) - T(x_2,y_1-1) + T(x_1-1,y_1-1)$                                                  |                                                                                               |
|                         | $avg = \frac{sum}{(x_2 - x_1 + 1)(y_2 - y_1 + 1)}$                                                                 |                                                                                               |
| **Storage Cost**        | > 2× original texture size (due to large numeric range).                                                          | ≈ 1.33× original texture size (sum of geometric series).                                      |
| **Rendering Time Work** | Must compute average during shading using SAT.                                                                    | Just a texture lookup — no math required per pixel.                                           |
| **Use with Texture Map**| Must replace normal lookup with averaged color from SAT.                                                          | Can continue standard texture mapping pipeline.                                               |
| **Advantages**          | - Arbitrary region support<br>- High accuracy                                                                     | - Very fast<br>- Hardware supported                                                           |
| **Drawbacks**           | - Needs large dynamic range<br>- Extra math during rendering                                                      | - Needs more texture memory<br>- May cause blur if mip levels not filtered well              |
| **Best For**            | - Custom filters<br>- Accurate shadow mapping                                                                     | - Standard 3D scenes<br>- Real-time applications                                              |